###1) Konfiguracja

In [0]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import uuid
import requests

BASE_URL = (
    "https://oedi-data-lake.s3.amazonaws.com/"
    "pvdaq/2023-solar-data-prize/2107_OEDI"
)

LANDING_PATH = Path(
    "/Volumes/workspace/energy_bronze/landing/nlr/system_2107"
)

FILES = [
    "metadata/2107_system_metadata.json",
    "data/2107_meter_15m_data.csv",
    "data/2107_meter_15m_data_2024.csv",
    "data/2107_environment_data.csv",
    "data/2107_environment_data_2024.csv",
    "data/2107_irradiance_data.csv",
    "data/2107_irradiance_data_2024.csv",
]

RUN_ID = str(uuid.uuid4())

LANDING_PATH.mkdir(parents=True, exist_ok=True)

print(f"Run ID: {RUN_ID}")
print(f"Folder docelowy: {LANDING_PATH}")
print(f"Liczba plików: {len(FILES)}")

###2) Pobranie plików

In [0]:
def calculate_sha256(file_path: Path) -> str:
    digest = hashlib.sha256()

    with file_path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(block)

    return digest.hexdigest()


session = requests.Session()
manifest_rows = []

for relative_path in FILES:
    source_url = f"{BASE_URL}/{relative_path}"
    target_path = LANDING_PATH / relative_path
    target_path.parent.mkdir(parents=True, exist_ok=True)

    head_response = session.head(source_url, timeout=30)
    head_response.raise_for_status()

    expected_size = int(
        head_response.headers.get("Content-Length", 0)
    )

    if (
        target_path.exists()
        and expected_size > 0
        and target_path.stat().st_size == expected_size
    ):
        status = "skipped_existing"

    else:
        temporary_path = Path(str(target_path) + ".part")
        temporary_path.unlink(missing_ok=True)

        with session.get(
            source_url,
            stream=True,
            timeout=(10, 180)
        ) as response:
            response.raise_for_status()

            with temporary_path.open("wb") as output_file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        output_file.write(chunk)

        downloaded_size = temporary_path.stat().st_size

        if expected_size > 0 and downloaded_size != expected_size:
            temporary_path.unlink(missing_ok=True)
            raise RuntimeError(
                f"Niepoprawny rozmiar {relative_path}: "
                f"{downloaded_size} zamiast {expected_size}"
            )

        os.replace(temporary_path, target_path)
        status = "downloaded"

    actual_size = target_path.stat().st_size

    manifest_rows.append({
        "ingestion_run_id": RUN_ID,
        "source_system": "NLR_PVDAQ",
        "system_id": 2107,
        "source_url": source_url,
        "relative_path": relative_path,
        "volume_path": str(target_path),
        "file_size_bytes": actual_size,
        "sha256": calculate_sha256(target_path),
        "ingestion_status": status,
        "source_etag": head_response.headers.get("ETag", ""),
        "ingested_at_utc": datetime.now(timezone.utc),
    })

    print(
        f"{status:18} "
        f"{actual_size / 1024 / 1024:8.2f} MB  "
        f"{relative_path}"
    )

###3) Zapis manifestu jako Delta Table

In [0]:
manifest_df = spark.createDataFrame(manifest_rows)

display(
    manifest_df.select(
        "ingestion_run_id",
        "relative_path",
        "file_size_bytes",
        "ingestion_status",
        "ingested_at_utc"
    )
)

(
    manifest_df.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(
        "workspace.energy_bronze.ingestion_manifest"
    )
)

###4) Kontrola plików

In [0]:
downloaded_files = [
    {
        "path": str(path),
        "size_mb": round(path.stat().st_size / 1024 / 1024, 2)
    }
    for path in sorted(LANDING_PATH.rglob("*"))
    if path.is_file()
]

display(spark.createDataFrame(downloaded_files))

assert len(downloaded_files) == len(FILES), (
    f"Oczekiwano {len(FILES)} plików, "
    f"znaleziono {len(downloaded_files)}"
)

print("Kontrola zakończona: wszystkie pliki są dostępne.")

###5) Podgląd metadanych farmy

In [0]:
metadata_path = (
    LANDING_PATH
    / "metadata"
    / "2107_system_metadata.json"
)

with metadata_path.open("r", encoding="utf-8") as file:
    metadata = json.load(file)

print("Sekcje pliku:")
print(list(metadata.keys()))

print("\nSYSTEM:")
print(json.dumps(metadata.get("System", {}), indent=2)[:4000])

print("\nSITE:")
print(json.dumps(metadata.get("Site", {}), indent=2)[:2000])

###6) Kontrola tabeli manifestu

In [0]:
%sql
SELECT
    ingestion_run_id,
    source_system,
    system_id,
    relative_path,
    ROUND(file_size_bytes / 1024 / 1024, 2) AS size_mb,
    ingestion_status,
    ingested_at_utc
FROM workspace.energy_bronze.ingestion_manifest
ORDER BY ingested_at_utc DESC;